# Custom Emulators — documentation examples

Companion notebook to the **Custom Emulators** documentation page
(`docs/source/custom_emulators.rst`). One section per code snippet.

Requires only the core `stellar-spice` install.

In [1]:
%matplotlib inline
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")

import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

## A worked example: Gaussian line on a blackbody continuum

In [2]:
from spice.spectrum import SpectrumEmulator
from spice.spectrum.blackbody import blackbody_intensity

class ToyLineEmulator(SpectrumEmulator):
    def __init__(self, line_center=5500.0, width=0.5, ld_coeff=0.6):
        self.line_center = line_center
        self.width = width
        self.ld_coeff = ld_coeff

    @property
    def stellar_parameter_names(self):
        return ["teff"]

    def to_parameters(self, parameters=None):
        parameters = parameters or {}
        return jnp.array([parameters.get("teff", 5777.0)])

    def intensity(self, log_wavelengths, mu, parameters):
        wavelengths = jnp.power(10.0, log_wavelengths)
        continuum = blackbody_intensity(log_wavelengths, mu, parameters)[:, 0]
        # linear limb darkening
        continuum = continuum * (1.0 - self.ld_coeff * (1.0 - mu))
        # deeper line for cooler atmospheres
        depth = jnp.clip(0.8 * 5777.0 / parameters[0], 0.0, 0.95)
        line = 1.0 - depth * jnp.exp(-0.5 * ((wavelengths - self.line_center) / self.width) ** 2)
        return jnp.stack([continuum * line, continuum], axis=-1)

emulator = ToyLineEmulator()

## Use it in synthesis

In [3]:
from spice.models import IcosphereModel
from spice.spectrum import simulate_observed_flux

star = IcosphereModel.construct(1000, 1., 1.,
                                emulator.to_parameters({"teff": 6000.}),
                                emulator.stellar_parameter_names)

wavelengths = np.linspace(5490., 5510., 800)
flux = simulate_observed_flux(emulator.intensity, star, np.log10(wavelengths))
print("shape:", flux.shape)

/Users/mjablons/code/spice/src/spice/models/mesh_model.py:331: UserWarning: If override_log_g is True, either parameter_names must include one of [logg,loggs,log_g,log_gs,log g,log gs,surface gravity,surface gravities,surface_gravity,surface_gravities], or log_g_index must be passed for log g to be used in the spectrum emulator.
  warnings.warn(f"If override_log_g is True, either parameter_names must include one of [" + ",".join(


[spice] IcosphereModel constructed in 0.6 s
shape: (800, 2)


## Everything composes

Rotation broadens the custom line; a temperature spot makes its depth
rotation-phase-dependent.

In [4]:
from spice.models.mesh_transform import add_rotation, evaluate_rotation
from spice.models.spots import add_spot

rotating = evaluate_rotation(add_rotation(star, rotation_velocity=40.), 0.)
flux_rot = simulate_observed_flux(emulator.intensity, rotating, np.log10(wavelengths))

spotted = add_spot(star, spot_center_theta=1.2, spot_center_phi=0.5,
                   spot_radius=30., parameter_delta=-1500., parameter_index=0)
flux_spot = simulate_observed_flux(emulator.intensity, spotted, np.log10(wavelengths))

fig, ax = plt.subplots(figsize=(9, 5))
for f, label in [(flux, "static"), (flux_rot, "40 km/s rotation"), (flux_spot, "cool spot")]:
    ax.plot(wavelengths, f[:, 0] / f[:, 1], label=label)
ax.set_xlabel(r"Wavelength [$\AA$]")
ax.set_ylabel("Normalized flux")
ax.legend()
plt.show()

/var/folders/7r/n_x0ntj511v_0gt816mgrc1c0000gq/T/ipykernel_74901/451204825.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
